# Spectral Pre-Analysis for MSI Apple Bruise Detection

This notebook performs spectral correlation analysis on 9-band MSI images to
understand the spectral differences between normal and bruised apple tissue.

**Outputs:**
- 9x9 Pearson correlation matrices for normal and bruise regions
- Difference heatmap (Delta-R)
- Mean spectral curves (normal vs bruise)

**Prerequisites:**
- Place your `.npy` images in `data/images/`
- Place your `.npy` masks in `data/masks/`

In [ ]:
import sys
sys.path.insert(0, '..')

import yaml
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from utils.spectral_analysis import (
    collect_spectral_data,
    compute_correlation_matrix,
    run_spectral_analysis,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

In [ ]:
# Load config
with open('../configs/config.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

data_cfg = cfg['data']
wavelengths = data_cfg.get('wavelengths', [713, 736, 759, 782, 805, 828, 851, 874, 897, 920])
print(f"Wavelengths: {wavelengths}")
print(f"Image dir: {data_cfg['image_dir']}")
print(f"Mask dir: {data_cfg['mask_dir']}")

In [ ]:
# Collect spectral data from all images
normal_spectra, bruise_spectra = collect_spectral_data(
    image_dir=data_cfg['image_dir'],
    mask_dir=data_cfg['mask_dir'],
)
print(f"Normal pixels: {normal_spectra.shape[0]:,}")
print(f"Bruise pixels: {bruise_spectra.shape[0]:,}")

In [ ]:
# Compute correlation matrices
R_normal = compute_correlation_matrix(normal_spectra)
R_bruise = compute_correlation_matrix(bruise_spectra)
delta_R = R_normal - R_bruise

wl_labels = [str(w) for w in wavelengths[:9]]

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for ax, mat, title, cmap, vmin in [
    (axes[0], R_normal[:9,:9], 'Normal Region', 'YlOrRd', 0),
    (axes[1], R_bruise[:9,:9], 'Bruise Region', 'YlOrRd', 0),
    (axes[2], delta_R[:9,:9], 'Delta-R (Normal - Bruise)', 'RdBu_r', -1),
]:
    sns.heatmap(mat, ax=ax, annot=True, fmt='.2f',
                xticklabels=wl_labels, yticklabels=wl_labels,
                cmap=cmap, vmin=vmin, vmax=1, square=True)
    ax.set_title(title)
    ax.set_xlabel('Wavelength (nm)')
    ax.set_ylabel('Wavelength (nm)')

plt.tight_layout()
plt.show()

In [ ]:
# Mean spectral curves
mean_normal = normal_spectra.mean(axis=0)
mean_bruise = bruise_spectra.mean(axis=0)
std_normal = normal_spectra.std(axis=0)
std_bruise = bruise_spectra.std(axis=0)

wl = np.array(wavelengths[:9])

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(wl, mean_normal[:9], 'b-o', label='Normal', markersize=6)
ax.fill_between(wl, mean_normal[:9]-std_normal[:9], mean_normal[:9]+std_normal[:9],
                alpha=0.2, color='blue')
ax.plot(wl, mean_bruise[:9], 'r-s', label='Bruise', markersize=6)
ax.fill_between(wl, mean_bruise[:9]-std_bruise[:9], mean_bruise[:9]+std_bruise[:9],
                alpha=0.2, color='red')

ax.set_xlabel('Wavelength (nm)', fontsize=12)
ax.set_ylabel('Reflectance', fontsize=12)
ax.set_title('Mean Spectral Curves: Normal vs Bruise Tissue', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Save all analysis results to outputs/spectral_analysis/
import os
output_dir = os.path.join(cfg['output']['root'], 'spectral_analysis')
results = run_spectral_analysis(
    image_dir=data_cfg['image_dir'],
    mask_dir=data_cfg['mask_dir'],
    output_dir=output_dir,
    wavelengths=wavelengths,
)
print(f"\nResults saved to {output_dir}")
print(f"Files: {os.listdir(output_dir)}")